# 🏛️ HUẤN LUYỆN MODEL QWEN3-14B CHO NGHIỆP VỤ HÀNH CHÍNH UBND CẤP XÃ**Phiên bản 2.0** — Nâng cấp từ Qwen2.5-3B lên Qwen3-14B-Instruct### Thay đổi chính:- **Model**: Qwen2.5-3B → **Qwen3-14B-Instruct** (chất lượng cao hơn đáng kể)- **Export**: Bỏ GGUF → **Safetensors merged_16bit** (tương thích ZeroGPU/transformers)- **Dataset**: 300 → **600 mẫu** (4 nhóm: trích xuất đơn, trích xuất bảng, phân công, soạn thảo)- **Địa danh**: Tổng quát hóa cho mọi UBND cấp Xã, không gắn cứng Cát Ngạn### Yêu cầu phần cứng:- Google Colab Pro (A100 40GB GPU) hoặc tương đương- ~20-30 phút huấn luyện với 600 mẫu, 3 epochs

## Bước 1: Cài đặt Thư viện Unsloth & Các Gói Hỗ Trợ

In [ ]:
# Cài đặt Unsloth phiên bản mới nhất hỗ trợ Qwen3!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo!pip install --no-deps trl peft accelerate bitsandbytes xformers!pip install datasets huggingface_hub

## Bước 2: Tải Base Model Qwen3-14B-Instruct (4-bit)

In [ ]:
import torchfrom unsloth import FastLanguageModel# Cấu hìnhMAX_SEQ_LENGTH = 4096DTYPE = None  # Tự động (Bfloat16 cho A100/H100)LOAD_IN_4BIT = True# Model nền tảng: Qwen3-14B-Instruct (Unsloth optimized)BASE_MODEL_NAME = "unsloth/Qwen3-14B-unsloth-bnb-4bit"print(f"🚀 Đang tải model: {BASE_MODEL_NAME}...")print(f"CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")model, tokenizer = FastLanguageModel.from_pretrained(    model_name=BASE_MODEL_NAME,    max_seq_length=MAX_SEQ_LENGTH,    dtype=DTYPE,    load_in_4bit=LOAD_IN_4BIT,)print("✅ Model đã tải thành công!")

## Bước 3: Thiết lập LoRA Adapters (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(    model,    r=16,    target_modules=[        "q_proj", "k_proj", "v_proj", "o_proj",        "gate_proj", "up_proj", "down_proj",    ],    lora_alpha=16,    lora_dropout=0,    bias="none",    use_gradient_checkpointing="unsloth",    random_state=3407,    use_rslora=False,    loftq_config=None,)print("✅ QLoRA Adapters đã được cấu hình!")

## Bước 4: Tạo / Nạp Tập Dữ Liệu Huấn LuyệnDataset gồm 600 mẫu chia 4 nhóm:1. **Trích xuất đơn** (150): OCR văn bản → JSON đơn2. **Trích xuất bảng** (150): Văn bản có bảng → JSON mảng (cho Excel)3. **Đề xuất phân công** (150): Phân tích cán bộ → JSON gợi ý4. **Soạn thảo** (150): Yêu cầu → Văn bản hành chính đúng thể thức

In [ ]:
import jsonimport osfrom datasets import load_dataset# Upload file dataset lên Colab hoặc kết nối Google Drive# Nếu dùng Google Drive:# from google.colab import drive# drive.mount('/content/drive')# DATASET_PATH = "/content/drive/MyDrive/ubnd_administrative_dataset.jsonl"# Nếu upload trực tiếp:from google.colab import filesuploaded = files.upload()  # Upload file ubnd_administrative_dataset.jsonlDATASET_PATH = "ubnd_administrative_dataset.jsonl"dataset = load_dataset("json", data_files={"train": DATASET_PATH}, split="train")print(f"✅ Đã nạp {len(dataset)} mẫu huấn luyện!")def formatting_prompts_func(examples):    convos = examples["messages"]    texts = [        tokenizer.apply_chat_template(            convo, tokenize=False, add_generation_prompt=False        )        for convo in convos    ]    return {"text": texts}dataset = dataset.map(formatting_prompts_func, batched=True)print("✅ Dataset đã được định dạng theo chat template!")

## Bước 5: Bắt Đầu Huấn Luyện (Fine-Tuning)

In [ ]:
from trl import SFTTrainerfrom transformers import TrainingArgumentsfrom unsloth import is_bfloat16_supportedOUTPUT_DIR = "outputs_qwen3_ubnd"trainer = SFTTrainer(    model=model,    tokenizer=tokenizer,    train_dataset=dataset,    dataset_text_field="text",    max_seq_length=MAX_SEQ_LENGTH,    dataset_num_proc=2,    packing=False,    args=TrainingArguments(        per_device_train_batch_size=2,        gradient_accumulation_steps=8,        warmup_steps=10,        num_train_epochs=3,        learning_rate=2e-4,        fp16=not is_bfloat16_supported(),        bf16=is_bfloat16_supported(),        logging_steps=10,        optim="adamw_8bit",        weight_decay=0.01,        lr_scheduler_type="cosine",        seed=3407,        output_dir=OUTPUT_DIR,        report_to="none",    ),)print("🚀 Bắt đầu huấn luyện...")trainer_stats = trainer.train()print(f"✅ Huấn luyện xong! Thời gian: {trainer_stats.metrics.get('train_runtime', 0):.0f} giây")

## Bước 6: Kiểm Thử Model Trực Tiếp

In [ ]:
FastLanguageModel.for_inference(model)test_messages = [    {"role": "system", "content": "Bạn là Trợ lý AI chuyên trách xử lý văn bản hành chính công vụ cho UBND cấp Xã theo chuẩn Nghị định 30/2020/NĐ-CP. Trả về JSON."},    {"role": "user", "content": "Phân tích văn bản: UBND Huyện yêu cầu UBND Xã báo cáo tình hình giải ngân vốn đầu tư công trước ngày 30/12/2026."}]input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)inputs = tokenizer(input_text, return_tensors="pt").to("cuda")with torch.no_grad():    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.2, do_sample=True)generated = outputs[0][inputs["input_ids"].shape[1]:]result = tokenizer.decode(generated, skip_special_tokens=True)print("📋 Kết quả phân tích:")print(result)

## Bước 7: Xuất Checkpoint Safetensors (Merged 16-bit)⚠️ **QUAN TRỌNG**: Không xuất GGUF nữa! ZeroGPU Space dùng `transformers` + `from_pretrained`.Xuất dạng **merged_16bit safetensors** — đây là định dạng chuẩn HuggingFace.

In [ ]:
import osMERGED_MODEL_DIR = "qwen3-14b-ubnd"os.makedirs(MERGED_MODEL_DIR, exist_ok=True)print("🔧 Đang merge LoRA adapters vào base model và xuất safetensors 16-bit...")model.save_pretrained_merged(    MERGED_MODEL_DIR,    tokenizer,    save_method="merged_16bit",)# Kiểm tra các file đã xuấtfiles = os.listdir(MERGED_MODEL_DIR)print(f"\n✅ Đã xuất {len(files)} files vào thư mục: {MERGED_MODEL_DIR}/")for f in sorted(files):    size_mb = os.path.getsize(os.path.join(MERGED_MODEL_DIR, f)) / (1024*1024)    print(f"  📄 {f} ({size_mb:.1f} MB)")

## Bước 8: Upload Checkpoint Lên HuggingFace Model RepoSau khi upload, cấu hình `MODEL_ID` trong ZeroGPU Space Settings trỏ tới repo này.

In [ ]:
from huggingface_hub import HfApi, login# Đăng nhập HuggingFace (cần Access Token có quyền Write)login()# Thay YOUR_USERNAME bằng tài khoản HuggingFace của bạnHF_REPO_ID = "YOUR_USERNAME/qwen3-14b-ubnd"api = HfApi()api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)print(f"📤 Đang upload checkpoint lên {HF_REPO_ID}...")api.upload_folder(    folder_path=MERGED_MODEL_DIR,    repo_id=HF_REPO_ID,    repo_type="model",)print(f"✅ Upload thành công! Model Repo: https://huggingface.co/{HF_REPO_ID}")print(f"\n👉 Bước tiếp theo:")print(f"   1. Vào ZeroGPU Space Settings → Variables")print(f"   2. Set MODEL_ID = {HF_REPO_ID}")print(f"   3. Restart Space để load model mới")